# Cell Type classification

This notebook focusses on cell type classification using the cell embeddings produced with scRepresenter by following the Training Model notebook. We are using Pbmc3k dataset from Scanpy and the corresponding cell type labels.

In [ ]:
# imports
import os
import pandas as pa
import scanpy as sc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
if os.path.basename(os.getcwd()) == 'scripts':
    os.chdir('..')

Start out by reading the embedding object produced by scRepresenter. This AnnData object contains the original processed Pbmc3k counts (see the Preprocessing notebook) as well as the embeddings of the scNET and scGPT models, and the averaged and concatenated hybrid embeddings of both. These embeddings will be used in a classification task to test the benefit of using scRepresenter.

In [ ]:
obj = sc.read("./output/pbmc3k/scRepresenter/Embs.h5ad")
print(obj.shape)
print(obj.obsm)

# Classification architecture

A standard transformer architecture is used as the cell type classifier.

In [ ]:
class Classifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(Classifier, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)

A standard 5-fold cross validation training loop is also used. It is important to save the predictions so we can add them to the AnnData object for later analysis.

In [ ]:
def test_embeddings(obj, epochs, name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    num_folds = 5
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    X = obj.obsm[name]
    y = obj.obs['celltype']
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)

    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    fold_losses = []

    all_y_true = []
    all_y_pred = []
    all_test_indices = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        all_test_indices.extend(test_idx)

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        #if method == "conq":
        #    pca = PCA(n_components=512)
        #    pca.fit(X_train)
        #    X_train = pca.transform(X_train)
        #    X_test = pca.transform(X_test)

        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.long)
        y_test_tensor = torch.tensor(y_test, dtype=torch.long)

        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_dataset, batch_size = 32, shuffle=True, drop_last=False)

        # Initialize model
        input_size = X_train.shape[1]
        num_classes = len(np.unique(y))
        model = Classifier(input_size, num_classes).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        # Train the model
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            correct_predictions = 0
            total_predictions = 0
            
            for batch_X, batch_y in train_loader:
                
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)

                _, predicted_labels = torch.max(outputs, 1)
                correct_predictions += (predicted_labels == batch_y).sum().item()
                total_predictions += batch_y.size(0)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
            
            accuracy = correct_predictions / total_predictions

            print(f"Epoch {epoch+1}/{epochs}, Training loss: {total_loss/len(train_loader):.4f}, Training accuracy: {accuracy:.4f}")


        model.eval()
        with torch.inference_mode():
            X_test_tensor = X_test_tensor.to(device)
            y_test_tensor = y_test_tensor.to(device)
            
            outputs = model(X_test_tensor)
            test_loss = criterion(outputs, y_test_tensor)

            _, predicted = torch.max(outputs, 1)
            y_true = y_test_tensor.cpu().numpy()
            y_pred = predicted.cpu().numpy()

            accuracy = accuracy_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred, average="macro", zero_division=1)
            recall = recall_score(y_true, y_pred, average="macro")
            f1 = f1_score(y_true, y_pred, average="macro")
            
            print(f"Fold {fold+1}: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1-Score={f1:.4f}")

            fold_losses.append(test_loss.item())
            fold_accuracies.append(accuracy)
            fold_precisions.append(precision)
            fold_recalls.append(recall)
            fold_f1s.append(f1)

            all_y_true.extend(y_true)
            all_y_pred.extend(y_pred)

    print("\nFinal Cross-Validation Results:")
    print(f"Mean Loss: {np.mean(fold_losses):.4f} ± {np.std(fold_losses):.4f}")
    print(f"Mean Accuracy: {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
    print(f"Mean Precision: {np.mean(fold_precisions):.4f} ± {np.std(fold_precisions):.4f}")
    print(f"Mean Recall: {np.mean(fold_recalls):.4f} ± {np.std(fold_recalls):.4f}")
    print(f"Mean F1-Score: {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")

    true_labels_decoded = label_encoder.inverse_transform(np.array(all_y_true))
    pred_labels_decoded = label_encoder.inverse_transform(np.array(all_y_pred))

    obj.obs[f"{name} prediction"] = None
    obj.obs.loc[obj.obs_names[all_test_indices], f"{name} prediction"] = pred_labels_decoded

    del model, optimizer, criterion
    torch.cuda.empty_cache()

# Classification and results

We can move on to the actual classification process, one for each of the embeddings. For an in-depth analysis of each see the Embedding and Prediction Analysis notebook. 

In [ ]:
#scNET
test_embeddings(obj, 5, 'X_scnet')

In [ ]:
#scGPT
test_embeddings(obj, 5, 'X_scgpt')

In [ ]:
#scRepresenter averaged
test_embeddings(obj, 5, 'X_combined_avg')

In [ ]:
#scRepresenter concatenated
test_embeddings(obj, 5, 'X_combined_conc')
obj.write("Predictions.h5ad")